# Experiment Analysis

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib notebook

In [ ]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path
from datetime import datetime as dt

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

import panel as pn

from enderleaf.const import (
    ImageMergeMode,
    TIME_FORMAT,
    PRECISE_TIME_FORMAT,
    DEFAULT_DATETIME_FORMAT,
    COLOR_SPACES,
)
from enderleaf.draw import image_grid, concat_tile_resize, plot_images_with_histograms
from enderleaf.tools import read_dataframe, write_dataframe, format_datetime
from enderleaf.image import (
    load_image,
    to_pil,
    canny,
    find_circles,
    filter_circles,
    crop_image,
    Rectangle,
    merge_images,
    merge_images_channels,
    match_previous_rotation,
    get_circles,
    get_channels,
    equalize_hist,
)
from enderleaf.draw import draw_circles

In [ ]:
pn.extension("ipywidgets")

## Constants

In [ ]:
EXP = "Exp00DM00"
INOC = "I0"
PLATE = 0
MONTH = 5
DAY = 26

PATH_TO_DATA = Path(".").joinpath("output", "job_data", EXP, INOC)
PATH_TO_IMAGES = Path(".").joinpath("output", "images", EXP, INOC)
MAX_CIRCLES=3

## Functions

In [ ]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [ ]:
df = pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")]).sort_values(
    ["plate", "row", "col"]
).dropna(subset="north")
# df = df[(df.plate == PLATE) & (df.month == MONTH) & (df.day == DAY)]
df["card_count"] = df[["north", "east", "west", "south"]].astype(int).sum(axis=1)
df["file_ok"] = df["file_name"].apply(lambda x:PATH_TO_IMAGES.joinpath(x).is_file())
df = df[df.file_ok == True]
df = df[df.job_ts != 20260515164717]
df["leaf_id"] = df.plate.astype(str)+df.row.astype(str)+df.col.astype(str)
df

In [ ]:
pd.DataFrame(df.groupby(["job_ts", "plate", "light_cycle"]).height.mean()).reset_index().sort_values("plate")

## Select Cycle ID

In [ ]:
sel_leaf_id = pn.widgets.Select(
    name="col",
    options=list(df[(df.job_ts == 20260526101311)].leaf_id.unique()),
    sizing_mode="scale_width",
)
sel_color_space = pn.widgets.Select(
    name="Color space",
    options=["rgb", "hsv", "lab", "yuv", "ycrcb"],
    sizing_mode="scale_width",
    value="rgb",
)
sel_channel_1 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][0],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_channel_2 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][1],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_channel_3 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][2],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
bt_random = pn.widgets.Button(name="Random Disc")

img_out = pn.pane.Matplotlib(sizing_mode="scale_width")


def on_random(event):
    row = df[["leaf_id"]].drop_duplicates().sample(n=1).iloc[0]
    sel_leaf_id.value = row.leaf_id


bt_random.on_click(on_random)


@pn.depends(sel_color_space.param.value, watch=True)
def on_color_space_changed(cs):
    sel_channel_1.name = COLOR_SPACES[cs][0]
    sel_channel_2.name = COLOR_SPACES[cs][1]
    sel_channel_3.name = COLOR_SPACES[cs][2]


@pn.depends(
    *[
        w.param.value
        for w in [
            sel_leaf_id,
            sel_color_space,
            sel_channel_1,
            sel_channel_2,
            sel_channel_3,
        ]
    ],
    watch=True,
)
def on_ld_changed(leaf_id, cs, cn1, cn2, cn3):
    df_ld = df[(df.leaf_id == leaf_id) & (df.job_ts == 20260526101311)]
    first_image = load(df_ld.iloc[0])
    height, width, _ = first_image.shape
    circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    merged_images = []
    light_cycles = []
    for light_cycle in df_ld.light_cycle.unique():
        merged_images.append(
            merge_images_channels(
                image_list=[
                    crop_image(load(row[1]), crop_data)
                    for row in df_ld[df_ld.light_cycle == light_cycle].iterrows()
                ],
                color_space=cs,
                merge_modes=(cn1, cn2, cn3),
            )
        )
        light_cycles.append(light_cycle)
    img_out.object = plot_images_with_histograms(
        images=merged_images, color_spaces=[cs, "rgb"], titles=light_cycles
    )


on_ld_changed(
    sel_leaf_id.value,
    sel_color_space.value,
    sel_channel_1.value,
    sel_channel_2.value,
    sel_channel_3.value,
)

pn.Column(
    pn.Row(
        sel_leaf_id,
        bt_random,
        sel_color_space,
        sel_channel_1,
        sel_channel_2,
        sel_channel_3,
    ),
    pn.Row(img_out),
)